In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
data = "fraud_detection_dataset_v2.csv"
df = pd.read_csv(data)
df.sample(5)

,transaction_id,timestamp,customer_id,merchant_id,transaction_type,amount,account_balance_before,account_balance_after,distance_from_home_km,new_payee,international_transaction,transactions_last_24h,failed_logins_last_week,merchant_category,device_trusted,ip_country,home_country,isFraud
26414,TXN-1026414,2026-01-19 04:46:48,CUST-41575,MERCH-2509,transfer,74.66,22772.90,22698.24,253.5,False,False,6,0,Apparel,True,US,US,0
38415,TXN-1038415,2026-01-27 11:46:17,CUST-53197,MERCH-9806,purchase,311.83,20040.26,19728.43,212.8,False,False,7,1,Luxury,True,AU,AU,0
38916,TXN-1038916,2026-01-27 19:27:28,CUST-57271,MERCH-5870,purchase,25.84,23806.09,23780.25,32.9,False,False,8,0,ATM,True,DE,DE,0
2530,TXN-1002530,2026-01-02 17:49:27,CUST-24662,MERCH-4550,purchase,184.02,5109.93,4925.91,23.6,False,False,3,0,Entertainment,True,JP,JP,0
15762,TXN-1015762,2026-01-11 20:44:31,CUST-61220,MERCH-3788,purchase,95.75,21622.18,21526.43,170.0,False,False,3,0,Entertainment,False,UK,UK,0


In [3]:
df.isnull().sum()

transaction_id               0
timestamp                    0
customer_id                  0
merchant_id                  0
transaction_type             0
amount                       0
account_balance_before       0
account_balance_after        0
distance_from_home_km        0
new_payee                    0
international_transaction    0
transactions_last_24h        0
failed_logins_last_week      0
merchant_category            0
device_trusted               0
ip_country                   0
home_country                 0
isFraud                      0
dtype: int64

In [4]:
df['isFraud'].value_counts()

isFraud
0    49251
1      749
Name: count, dtype: int64

In [5]:
df.groupby("international_transaction")["isFraud"].mean()

international_transaction
False    0.013408
True     0.029275
Name: isFraud, dtype: float64

In [6]:
df.groupby("new_payee")["isFraud"].mean()

new_payee
False    0.013202
True     0.028047
Name: isFraud, dtype: float64

In [7]:
df.groupby("device_trusted")["isFraud"].mean()

device_trusted
False    0.020735
True     0.013984
Name: isFraud, dtype: float64

In [8]:
df.groupby("international_transaction")["isFraud"].agg(["count","mean"])

,count,mean
international_transaction,,
False,45047,0.013408
True,4953,0.029275


In [9]:
pd.crosstab(
    [df["international_transaction"], df["new_payee"]],
    df["isFraud"],
    normalize="index"
)

isFraud                                     0         1
international_transaction new_payee                    
False                     False      0.986574  0.013426
                          True       0.986718  0.013282
True                      False      0.988823  0.011177
                          True       0.831283  0.168717

In [10]:
df["amount_balance_ratio"] = (
    df["amount"] / df["account_balance_before"]
)

In [11]:
df["country_mismatch"] = (
    df["ip_country"] != df["home_country"]
).astype(int)

In [12]:
df.groupby('country_mismatch')['isFraud'].mean()

country_mismatch
0    0.013408
1    0.029275
Name: isFraud, dtype: float64

In [13]:
df["high_velocity"] = (
    df["transactions_last_24h"] > 7
).astype(int)

In [14]:
df.groupby('high_velocity')['isFraud'].mean()

high_velocity
0    0.015077
1    0.014593
Name: isFraud, dtype: float64

In [15]:
df.groupby("high_velocity")["isFraud"].agg(["count","mean"])

,count,mean
high_velocity,,
0,39995,0.015077
1,10005,0.014593


In [16]:
(df["country_mismatch"] == df["international_transaction"]).mean()

1.0

In [17]:
df["weekend_transaction"] = (
    pd.to_datetime(df["timestamp"])
      .dt.dayofweek
      .isin([5,6])
).astype(int)

In [18]:
df["hour"] = pd.to_datetime(df["timestamp"]).dt.hour

df["night_transaction"] = (
    df["hour"].isin([0,1,2,3,4])
).astype(int)

In [19]:
df['velocity_amount_risk'] = df['transactions_last_24h'] * df['amount']

In [20]:
features = [
    "amount",
    "account_balance_before",
    "account_balance_after",
    "distance_from_home_km",
    "device_trusted",
    "new_payee",
    "international_transaction",
    "transactions_last_24h",
    "failed_logins_last_week",
    "merchant_category",
    "transaction_type",
    "velocity_amount_risk",
    "amount_balance_ratio"
]

In [68]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    roc_auc_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report
)


In [69]:
X = df[features]
y = df["isFraud"]

In [70]:
X = pd.get_dummies(
    X,
    columns=[
        "merchant_category",
        "transaction_type",
    ],
    drop_first=True
)

In [71]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)
rf = RandomForestClassifier(
    n_estimators=200,
    max_depth=10,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)

rf.fit(X_train, y_train)


RandomForestClassifier(class_weight='balanced', max_depth=10, n_estimators=200,
                       n_jobs=-1, random_state=42)

In [72]:

probs = rf.predict_proba(X_test)[:, 1]

preds = (probs >= 0.40).astype(int)

In [73]:
print("ROC-AUC  :", roc_auc_score(y_test, probs))
print("Precision:", precision_score(y_test, preds))
print("Recall   :", recall_score(y_test, preds))
print("F1 Score :", f1_score(y_test, preds))

print("\nClassification Report")
print(classification_report(y_test, preds))

ROC-AUC  : 0.6689576988155668
Precision: 0.055323590814196244
Recall   : 0.35333333333333333
F1 Score : 0.09566787003610108

Classification Report
              precision    recall  f1-score   support

           0       0.99      0.91      0.95      9850
           1       0.06      0.35      0.10       150

    accuracy                           0.90     10000
   macro avg       0.52      0.63      0.52     10000
weighted avg       0.98      0.90      0.93     10000



In [74]:
from xgboost import XGBClassifier
from sklearn.metrics import (
    roc_auc_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report
)

# Calculate class imbalance
neg = (y_train == 0).sum()
pos = (y_train == 1).sum()

xgb = XGBClassifier(
    n_estimators=300,
    max_depth=5,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    eval_metric="logloss"
)

xgb.fit(X_train, y_train)

probs = xgb.predict_proba(X_test)[:, 1]


preds = (probs >= 0.01).astype(int)

print("ROC-AUC  :", roc_auc_score(y_test, probs))
print("Precision:", precision_score(y_test, preds))
print("Recall   :", recall_score(y_test, preds))
print("F1 Score :", f1_score(y_test, preds))

print("\nClassification Report")
print(classification_report(y_test, preds))

ROC-AUC  : 0.7030490693739425
Precision: 0.02617801047120419
Recall   : 0.6333333333333333
F1 Score : 0.05027785128340831

Classification Report
              precision    recall  f1-score   support

           0       0.99      0.64      0.78      9850
           1       0.03      0.63      0.05       150

    accuracy                           0.64     10000
   macro avg       0.51      0.64      0.41     10000
weighted avg       0.98      0.64      0.77     10000



In [75]:
for t in [0.2, 0.25, 0.3, 0.35, 0.4]:
    preds = (probs >= t).astype(int)

    print(
        t,
        precision_score(y_test, preds),
        recall_score(y_test, preds),
        f1_score(y_test, preds)
    )

0.2 0.5789473684210527 0.29333333333333333 0.3893805309734513
0.25 0.6231884057971014 0.2866666666666667 0.3926940639269406
0.3 0.6461538461538462 0.28 0.39069767441860465
0.35 0.7017543859649122 0.26666666666666666 0.3864734299516908
0.4 0.6730769230769231 0.23333333333333334 0.3465346534653465


In [76]:
for t in [0.45, 0.5, 0.55, 0.6]:
    preds = (probs >= t).astype(int)

    print(
        t,
        precision_score(y_test, preds),
        recall_score(y_test, preds),
        f1_score(y_test, preds)
    )

0.45 0.6938775510204082 0.22666666666666666 0.3417085427135678
0.5 0.6976744186046512 0.2 0.31088082901554404
0.55 0.6904761904761905 0.19333333333333333 0.3020833333333333
0.6 0.6666666666666666 0.13333333333333333 0.2222222222222222


In [77]:
importance = pd.Series(
    xgb.feature_importances_,
    index=X.columns
).sort_values(ascending=False)

print(importance.head(15))

transaction_type_transfer          0.164283
new_payee                          0.136851
device_trusted                     0.111562
failed_logins_last_week            0.079795
international_transaction          0.076464
amount                             0.039990
distance_from_home_km              0.039983
velocity_amount_risk               0.036556
transactions_last_24h              0.034609
merchant_category_Grocery          0.029658
account_balance_before             0.027714
account_balance_after              0.027247
amount_balance_ratio               0.027108
transaction_type_withdrawal        0.026995
merchant_category_Entertainment    0.026495
dtype: float32


In [78]:
from lightgbm import LGBMClassifier
from sklearn.metrics import (
    roc_auc_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report
)

lgbm = LGBMClassifier(
    n_estimators=300,
    learning_rate=0.05,
    num_leaves=31,
    random_state=42,
    class_weight="balanced",
    verbose=-1
)

lgbm.fit(X_train, y_train)

probs = lgbm.predict_proba(X_test)[:, 1]

for t in [0.2, 0.25, 0.3, 0.35, 0.4]:
    preds = (probs >= t).astype(int)

    print(
        f"Threshold={t}",
        "Precision=", round(precision_score(y_test, preds), 3),
        "Recall=", round(recall_score(y_test, preds), 3),
        "F1=", round(f1_score(y_test, preds), 3)
    )

# Choose best threshold after inspecting results
preds = (probs >= 0.2).astype(int)

print("\nROC-AUC  :", roc_auc_score(y_test, probs))
print("Precision:", precision_score(y_test, preds))
print("Recall   :", recall_score(y_test, preds))
print("F1 Score :", f1_score(y_test, preds))

print("\nClassification Report")
print(classification_report(y_test, preds))

Threshold=0.2 Precision= 0.029 Recall= 0.62 F1= 0.056
Threshold=0.25 Precision= 0.038 Recall= 0.56 F1= 0.07
Threshold=0.3 Precision= 0.049 Recall= 0.507 F1= 0.09
Threshold=0.35 Precision= 0.067 Recall= 0.46 F1= 0.117
Threshold=0.4 Precision= 0.086 Recall= 0.393 F1= 0.141

ROC-AUC  : 0.7158443316412859
Precision: 0.029346797096875987
Recall   : 0.62
F1 Score : 0.056040976197649896

Classification Report
              precision    recall  f1-score   support

           0       0.99      0.69      0.81      9850
           1       0.03      0.62      0.06       150

    accuracy                           0.69     10000
   macro avg       0.51      0.65      0.43     10000
weighted avg       0.98      0.69      0.80     10000



In [1]:
from catboost import CatBoostClassifier
from sklearn.metrics import *

scale_weight = 40

cat = CatBoostClassifier(
    iterations=1000,
    depth=6,
    learning_rate=0.05,
    loss_function="Logloss",
    eval_metric="AUC",
    scale_pos_weight=scale_weight,  # <-- THIS IS CRITICAL TO FIX THE RECALL
    random_seed=42,
    verbose=100
)

cat.fit(X_train, y_train)

probs = cat.predict_proba(X_test)[:, 1]

print("ROC-AUC:", roc_auc_score(y_test, probs))
preds = (probs >= 0.4).astype(int)

print("PR-AUC       :", average_precision_score(y_test, probs))
print("Precision    :", precision_score(y_test, preds))
print("Recall       :", recall_score(y_test, preds))
print("F1 Score     :", f1_score(y_test, preds))

print("\nConfusion Matrix")
print(confusion_matrix(y_test, preds))

ModuleNotFoundError: No module named 'catboost'

In [80]:
for t in [0.05,0.1, 0.15, 0.2, 0.25, 0.3, 0.35, 0.4]:
    preds = (probs >= t).astype(int)

    print(
        t,
        precision_score(y_test, preds),
        recall_score(y_test, preds),
        f1_score(y_test, preds),
        roc_auc_score(y_test, probs),
        average_precision_score(y_test, probs)
    )

0.05 0.02084528590552687 0.7266666666666667 0.040527979178285926 0.7008561759729272 0.22529331644352876
0.1 0.028455284552845527 0.56 0.05415860735009671 0.7008561759729272 0.22529331644352876
0.15 0.04044321329639889 0.4866666666666667 0.07468030690537085 0.7008561759729272 0.22529331644352876
0.2 0.05714285714285714 0.4266666666666667 0.10078740157480315 0.7008561759729272 0.22529331644352876
0.25 0.07810320781032078 0.37333333333333335 0.12918108419838523 0.7008561759729272 0.22529331644352876
0.3 0.1111111111111111 0.3466666666666667 0.16828478964401294 0.7008561759729272 0.22529331644352876
0.35 0.15457413249211358 0.32666666666666666 0.20985010706638116 0.7008561759729272 0.22529331644352876
0.4 0.20535714285714285 0.30666666666666664 0.24598930481283424 0.7008561759729272 0.22529331644352876


In [81]:
frauds = X_test.copy()
frauds["actual"] = y_test
frauds["prob"] = probs

missed = frauds[
    (frauds["actual"] == 1) &
    (frauds["prob"] < 0.2)
]

print(missed.head(20))

       amount  account_balance_before  account_balance_after  \
44182    4.73                11032.98               11028.25   
30664  138.64                18089.87               17951.23   
24876  122.24                14194.89               14072.65   
41848   79.77                13818.94               13739.17   
18344   47.79                14511.18               14463.39   
21174   98.71                 6684.71                6586.00   
28652  133.49                  271.46                 137.97   
43148   28.35                 7750.14                7721.79   
46976  145.33                14098.34               13953.01   
25184    4.47                21676.38               21671.91   
25133   25.46                20306.84               20281.38   
1674    22.62                11252.63               11230.01   
32574  156.24                 6384.36                6228.12   
17976  119.84                 1716.69                1596.85   
10807   48.04                 3336.84   

In [89]:
# Save using native CatBoost method instead of joblib
cat.save_model("financial_fraud_model.cbm")

In [83]:
df.groupby("transaction_type")['isFraud'].mean()

transaction_type
purchase      0.012887
transfer      0.023109
withdrawal    0.013381
Name: isFraud, dtype: float64

In [84]:
# Find out what the maximum score the model gives to normal transactions is
normal_scores = cat.predict_proba(X_test[y_test == 0])[:, 1]
fraud_scores = cat.predict_proba(X_test[y_test == 1])[:, 1]

print(f"95th percentile of normal transactions: {np.percentile(normal_scores, 95)}")
print(f"Median score for actual fraud: {np.median(fraud_scores)}")

95th percentile of normal transactions: 0.280822236122494
Median score for actual fraud: 0.146846954970517


In [85]:
from sklearn.metrics import roc_auc_score, classification_report
import pandas as pd

# 1. Run predictions on your test set
test_probs = cat.predict_proba(X_test)[:, 1]

# 2. Check the AUC-ROC Score
auc = roc_auc_score(y_test, test_probs)
print(f"CRITICAL CHECK - AUC-ROC Score: {auc}")

# 3. Print full distribution descriptions
print("\n--- Clean Transactions Distribution ---")
print(pd.Series(cat.predict_proba(X_test[y_test == 0])[:, 1]).describe())

print("\n--- Fraud Transactions Distribution ---")
print(pd.Series(cat.predict_proba(X_test[y_test == 1])[:, 1]).describe())

CRITICAL CHECK - AUC-ROC Score: 0.7008561759729272

--- Clean Transactions Distribution ---
count    9850.000000
mean        0.086562
std         0.102000
min         0.000021
25%         0.020759
50%         0.052936
75%         0.114324
max         0.995734
dtype: float64

--- Fraud Transactions Distribution ---
count    150.000000
mean       0.321031
std        0.349228
min        0.000058
25%        0.047616
50%        0.146847
75%        0.662627
max        0.991200
dtype: float64


In [86]:
df.groupby('weekend_transaction')['isFraud'].mean()

weekend_transaction
0    0.015841
1    0.012902
Name: isFraud, dtype: float64